**Import Libraries**

In [1]:
import json
from pathlib import Path

import joblib
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    average_precision_score,
    classification_report
)


**Load Dataset**

In [2]:
df = pd.read_csv("data.csv")

In [3]:
FEATURES = [
    "Air temperature [K]",
    "Process temperature [K]",
    "Rotational speed [rpm]",
    "Torque [Nm]",
    "Tool wear [min]"
]

In [4]:
TARGET = "Machine failure"


In [5]:
X = df[FEATURES].copy()
y = df[TARGET].copy()

assert X.notna().all().all(), "Missing feature values detected"
assert y.notna().all(), "Missing target labels detected"
assert set(y.unique()).issubset({0, 1}), "Target must be binary"

print("Dataset:", X.shape)
print("Failure distribution:")
print(y.value_counts(normalize=True))

Dataset: (10000, 5)
Failure distribution:
Machine failure
0    0.9661
1    0.0339
Name: proportion, dtype: float64


**Train - Test - Validation split**

In [6]:

X_train, X_temp, y_train, y_temp = train_test_split(
    X,
    y,
    test_size=0.30,
    random_state=42,
    stratify=y
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.50,
    random_state=42,
    stratify=y_temp
)

assert X_train.index.equals(y_train.index)
assert X_val.index.equals(y_val.index)
assert X_test.index.equals(y_test.index)

assert not (set(X_train.index) & set(X_val.index))
assert not (set(X_train.index) & set(X_test.index))
assert not (set(X_val.index) & set(X_test.index))

print("Train:", X_train.shape)
print("Validation:", X_val.shape)
print("Test:", X_test.shape)


Train: (7000, 5)
Validation: (1500, 5)
Test: (1500, 5)


Define Candidate Models

1.   Standard Logistic Regression
2.   Balanced Logistic Regression
1.   Random Forest Classifier


In [7]:

models = {
    "Logistic Regression": Pipeline([
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(
            random_state=42,
            max_iter=1000
        ))
    ]),

    "Balanced Logistic Regression": Pipeline([
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(
            random_state=42,
            max_iter=1000,
            class_weight="balanced"
        ))
    ]),

    "Random Forest": Pipeline([
        ("model", RandomForestClassifier(
            n_estimators=100,
            random_state=42,
            n_jobs=-1
        ))
    ])
}


**Train & Evaluate on Validation Data**

In [8]:
validation_results = []

for name, model in models.items():

    model.fit(X_train, y_train)

    y_pred = model.predict(X_val)
    y_prob = model.predict_proba(X_val)[:, 1]

    metrics = {
        "model": name,
        "accuracy": accuracy_score(y_val, y_pred),
        "precision": precision_score(
            y_val, y_pred, zero_division=0
        ),
        "recall": recall_score(
            y_val, y_pred, zero_division=0
        ),
        "f1": f1_score(
            y_val, y_pred, zero_division=0
        ),
        "pr_auc": average_precision_score(
            y_val, y_prob
        )
    }

    validation_results.append(metrics)

    print(f"\n{name}")
    print(pd.Series(metrics))
    print("Confusion matrix:")
    print(confusion_matrix(y_val, y_pred))

results_df = pd.DataFrame(validation_results)

print("\nVALIDATION COMPARISON")
print(results_df.to_string(index=False))


Logistic Regression
model        Logistic Regression
accuracy                0.969333
precision               0.727273
recall                  0.156863
f1                      0.258065
pr_auc                   0.41496
dtype: object
Confusion matrix:
[[1446    3]
 [  43    8]]

Balanced Logistic Regression
model        Balanced Logistic Regression
accuracy                            0.834
precision                        0.136029
recall                            0.72549
f1                               0.229102
pr_auc                           0.403714
dtype: object
Confusion matrix:
[[1214  235]
 [  14   37]]

Random Forest
model        Random Forest
accuracy             0.986
precision         0.857143
recall            0.705882
f1                0.774194
pr_auc            0.782686
dtype: object
Confusion matrix:
[[1443    6]
 [  15   36]]

VALIDATION COMPARISON
                       model  accuracy  precision   recall       f1   pr_auc
         Logistic Regression  0.969333   0.72

**Select model using validation F1**

In [9]:
best_row = results_df.sort_values(
    "f1", ascending=False
).iloc[0]

best_name = best_row["model"]
best_model = models[best_name]

print("\nSelected model:", best_name)
print("Validation F1:", best_row["f1"])


Selected model: Random Forest
Validation F1: 0.7741935483870968



All candidates were already fitted on X_train.
Do not fit the selected model on X_test.

**Final held-out test evaluation**

In [10]:
test_pred = best_model.predict(X_test)
test_prob = best_model.predict_proba(X_test)[:, 1]

print("\nFINAL TEST RESULTS")
print(classification_report(
    y_test, test_pred, zero_division=0
))
print("Confusion matrix:")
print(confusion_matrix(y_test, test_pred))
print(
    "Test PR-AUC:",
    average_precision_score(y_test, test_prob)
)



FINAL TEST RESULTS
              precision    recall  f1-score   support

           0       0.99      1.00      0.99      1449
           1       0.88      0.57      0.69        51

    accuracy                           0.98      1500
   macro avg       0.93      0.78      0.84      1500
weighted avg       0.98      0.98      0.98      1500

Confusion matrix:
[[1445    4]
 [  22   29]]
Test PR-AUC: 0.7913065104964094


**Save Model & Metadata**

In [11]:
artifact_dir = Path("artifacts")
artifact_dir.mkdir(exist_ok=True)

joblib.dump(
    best_model,
    artifact_dir / "factorymind_model.joblib"
)

metadata = {
    "model_name": best_name,
    "features": FEATURES,
    "target": TARGET,
    "decision_threshold": 0.5,
    "validation_f1": float(best_row["f1"]),
    "dataset": "AI4I 2020 synthetic dataset",
    "random_state": 42
}

with open(
    artifact_dir / "model_metadata.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(metadata, f, indent=2)

**Verify Saved Model Data**

In [12]:
loaded_model = joblib.load(
    artifact_dir / "factorymind_model.joblib"
)

sample = X_test.iloc[[0]][FEATURES]

print("\nSample prediction:")
print("Predicted class:", loaded_model.predict(sample)[0])
print(
    "Failure probability:",
    loaded_model.predict_proba(sample)[0, 1]
)

print("\nTraining pipeline completed.")



Sample prediction:
Predicted class: 0
Failure probability: 0.0

Training pipeline completed.
